# 04 — BERT Baseline Training & Cross-Domain Zero-Shot Evaluation


1. **Baseline Training** — fine-tune `bert-base-cased` on CoNLL-2003, source domain.
2. **Baseline Evaluation** — resolve the label-space-mismatch

**On that design decision:** we're going with the entity-boundary-only approach collapsing every entity type to one generic "is this an entity" class before scoring. It requires no manual category-mapping judgment calls and is defensible on its own.

In [1]:
!pip -q install "transformers==4.44.2" "datasets==2.19.2" "seqeval==1.2.2" "accelerate>=0.26.0"
print("Done installing.")

import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected -- go to Runtime > Change runtime type and select a GPU before continuing.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 73.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.1/542.1 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 67.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.3.1 which is incompatible.
Done installing.
GPU a

## Step 1 — Load the tokenized data and label map



In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import json
from pathlib import Path
from datasets import load_from_disk

PROCESSED = Path('/content/drive/MyDrive/AAI590/data/processed')
TOKENIZED_DIR = PROCESSED / "tokenized"
LABELS_DIR = PROCESSED / "label_maps"

conll_train = load_from_disk(str(TOKENIZED_DIR / "conll2003" / "train"))
conll_val = load_from_disk(str(TOKENIZED_DIR / "conll2003" / "validation"))
conll_test = load_from_disk(str(TOKENIZED_DIR / "conll2003" / "test"))

with open(LABELS_DIR / "conll2003_label_map.json") as f:
    conll_label_map = json.load(f)

# JSON always saves dict keys as strings, so id2label's integer keys need converting back
conll_id2label = {int(k): v for k, v in conll_label_map["id2label"].items()}
conll_label2id = conll_label_map["label2id"]
num_labels = len(conll_id2label)

print("Train / Val / Test sizes:", len(conll_train), len(conll_val), len(conll_test))
print("Num labels:", num_labels)
print("Labels:", conll_id2label)


Mounted at /content/drive
Train / Val / Test sizes: 14041 3250 3453
Num labels: 8
Labels: {0: 'O', 1: 'B-LOC', 2: 'B-MISC', 3: 'B-ORG', 4: 'I-LOC', 5: 'I-MISC', 6: 'I-ORG', 7: 'I-PER'}


## Step 1.5 — Verify nothing is getting silently truncated

`03_tokenize_and_align.ipynb` truncated every sentence to 256 subword tokens. That's almost certainly enough for CoNLL-2003 (average sentence length 14.5 words), but SciERC had a long tail of much longer sentences in the EDA histogram. If any sentence actually needs more than 256 subwords, the excess gets silently cut off — any entity sitting in that cut-off tail disappears from training with no error or warning. Worth checking for certain rather than assuming.


In [3]:
def check_truncation(dataset, name, max_length=256):
    lengths = [len(ex["input_ids"]) for ex in dataset]
    n_at_limit = sum(1 for l in lengths if l >= max_length)
    print(f"{name}: max length = {max(lengths)}, "
          f"{n_at_limit} / {len(lengths)} sequences hit the {max_length}-token limit "
          f"({100 * n_at_limit / len(lengths):.2f}%)")
    return n_at_limit

for ds, name in [(conll_train, "conll2003/train"), (conll_val, "conll2003/validation"), (conll_test, "conll2003/test")]:
    check_truncation(ds, name)

print()
print("If any dataset shows sequences hitting the limit, re-run 03_tokenize_and_align.ipynb")
print("with a larger max_length in tokenize_and_align_labels() before training on it --")
print("otherwise those sentences are silently losing their tail-end entities right now.")


conll2003/train: max length = 173, 0 / 14041 sequences hit the 256-token limit (0.00%)
conll2003/validation: max length = 151, 0 / 3250 sequences hit the 256-token limit (0.00%)
conll2003/test: max length = 148, 0 / 3453 sequences hit the 256-token limit (0.00%)

If any dataset shows sequences hitting the limit, re-run 03_tokenize_and_align.ipynb
with a larger max_length in tokenize_and_align_labels() before training on it --
otherwise those sentences are silently losing their tail-end entities right now.


## Step 1.6 — Set a random seed

Training isn't currently seeded, so re-running this notebook would give a slightly different model each time (weight initialization for the classification head, batch shuffling order, etc.). Setting a seed makes the run reproducible.


In [4]:
from transformers import set_seed
set_seed(42)


## Step 2 — Load the model

`AutoModelForTokenClassification` adds a classification head on top of BERT sized to `num_labels` — one output per entity tag, applied independently to every token position. Passing `id2label`/`label2id` in means the model's config remembers what each output number means, so predictions come back as readable tags later instead of bare integers.


In [5]:
from transformers import AutoModelForTokenClassification

MODEL_NAME = "bert-base-cased"

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=conll_id2label,
    label2id=conll_label2id,
)
print(model.config.num_labels, "output classes")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


8 output classes


## Step 3 — Data collator: dynamic padding

Sentences in a batch aren't all the same length, but tensors need to be rectangular. `DataCollatorForTokenClassification` pads every batch to the length of its *longest* sentence in that batch (not a fixed max length for the whole dataset), which is both correct and more efficient than padding everything to 256 tokens up front. It also knows to pad the `labels` field with `-100` (not `0`), so padded positions are correctly ignored in the loss, same as the alignment logic from the last notebook.


In [6]:
from transformers import AutoTokenizer, DataCollatorForTokenClassification

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


## Step 4 — Define the evaluation metric

NER is scored at the **entity span** level, not per-token — getting 8 out of 9 tokens of a long entity right still counts as a miss for that entity, not partial credit. The `seqeval` library implements this correctly given BIO-tagged sequences, which is why it's the standard choice for NER (and matches the "F1 Score" metric named on your Methodology slide).


In [7]:
import numpy as np
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)

    true_labels, true_predictions = [], []
    for pred_row, label_row in zip(predictions, labels):
        sent_labels, sent_preds = [], []
        for p, l in zip(pred_row, label_row):
            if l == -100:
                continue  # skip special tokens and continuation subwords
            sent_labels.append(conll_id2label[l])
            sent_preds.append(conll_id2label[p])
        true_labels.append(sent_labels)
        true_predictions.append(sent_preds)

    return {
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions),
    }


## Step 5 — Train

Standard fine-tuning hyperparameters for BERT-base on a dataset this size: learning rate `2e-5`, 3 epochs, batch size 16. `load_best_model_at_end=True` means the checkpoint you end up with is whichever epoch had the best validation F1, not necessarily the last one.


In [8]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="/content/baseline_conll2003_checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=conll_train,
    eval_dataset=conll_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.139300,0.033335,0.936660,0.938236,0.937447
2,0.025500,0.035286,0.942498,0.946146,0.944318
3,0.012800,0.032354,0.947695,0.951363,0.949525


TrainOutput(global_step=2634, training_loss=0.04516208257834479, metrics={'train_runtime': 551.7738, 'train_samples_per_second': 76.341, 'train_steps_per_second': 4.774, 'total_flos': 1050525062669376.0, 'train_loss': 0.04516208257834479, 'epoch': 3.0})

## Step 6 — Evaluate on the held-out CoNLL-2003 test set

This is the **in-domain** number — how good the baseline is on the same kind of text it was trained on. It's the reference point everything else (cross-domain zero-shot below, and later the fine-tuned/DAP models) gets compared against.


In [9]:
test_results = trainer.evaluate(conll_test)
print("CoNLL-2003 test set (in-domain):")
for k, v in test_results.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")


CoNLL-2003 test set (in-domain):
  eval_loss: 0.1073
  eval_precision: 0.9041
  eval_recall: 0.9168
  eval_f1: 0.9104
  eval_runtime: 11.6115
  eval_samples_per_second: 297.3780
  eval_steps_per_second: 9.3010
  epoch: 3.0000


In [10]:
# Full per-entity-type breakdown, not just the overall F1
predictions, labels, _ = trainer.predict(conll_test)
predictions = np.argmax(predictions, axis=2)

true_labels, true_predictions = [], []
for pred_row, label_row in zip(predictions, labels):
    sent_labels, sent_preds = [], []
    for p, l in zip(pred_row, label_row):
        if l == -100:
            continue
        sent_labels.append(conll_id2label[l])
        sent_preds.append(conll_id2label[p])
    true_labels.append(sent_labels)
    true_predictions.append(sent_preds)

print(classification_report(true_labels, true_predictions, digits=3))


              precision    recall  f1-score   support

         LOC      0.929     0.931     0.930      1668
        MISC      0.754     0.809     0.781       702
         ORG      0.886     0.908     0.897      1661
         PER      0.968     0.958     0.963      1617

   micro avg      0.904     0.917     0.910      5648
   macro avg      0.884     0.902     0.893      5648
weighted avg      0.906     0.917     0.911      5648



## Step 7 — Save the trained model to Drive




In [11]:
MODELS_DIR = PROCESSED / "models"
baseline_dir = MODELS_DIR / "baseline_conll2003"
baseline_dir.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(baseline_dir))
tokenizer.save_pretrained(str(baseline_dir))

print("Saved baseline model to:", baseline_dir)
print("Contents:", sorted(p.name for p in baseline_dir.glob("*")))


Saved baseline model to: /content/drive/MyDrive/AAI590/data/processed/models/baseline_conll2003
Contents: ['config.json', 'model.safetensors', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json', 'training_args.bin', 'vocab.txt']


## Step 8 — Cross-domain zero-shot evaluation (entity-boundary F1)

Now the part flagged in the last notebook: how well does this CoNLL-only model do on WNUT-17 and SciERC, where it has never seen the real label types?

**The key idea:** the model can only ever output CoNLL's 8 labels — it has no `B-Method` or `B-corporation` slot to predict, no matter what the true answer is. So instead of asking "did it get the exact entity type right" (a question the model was never equipped to answer), we ask a fairer question: **"did it correctly recognize that a span of text is an entity at all, regardless of type?"** That means collapsing every `B-*` tag to a single `B-ENT` and every `I-*` tag to `I-ENT`, for both the model's predictions and the true labels, before scoring.


In [12]:
def collapse_to_boundary(tag):
    if tag == "O":
        return "O"
    prefix = tag[0]  # 'B' or 'I'
    return f"{prefix}-ENT"

def predict_labels(model, tokenizer, dataset, id2label, batch_size=32):
    """Runs the model over a tokenized dataset and returns predicted tag strings per sentence,
    skipping any position where the true label was -100 (special tokens / continuation subwords)."""
    from torch.utils.data import DataLoader
    import torch

    model.eval()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)

    collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
    loader = DataLoader(dataset, batch_size=batch_size, collate_fn=collator)

    all_true, all_pred = [], []
    with torch.no_grad():
        for batch in loader:
            labels = batch.pop("labels")
            batch = {k: v.to(device) for k, v in batch.items()}
            logits = model(**batch).logits
            preds = torch.argmax(logits, dim=-1).cpu()

            for pred_row, label_row in zip(preds, labels):
                sent_true, sent_pred = [], []
                for p, l in zip(pred_row.tolist(), label_row.tolist()):
                    if l == -100:
                        continue
                    sent_true.append(l)
                    sent_pred.append(p)
                all_true.append(sent_true)
                all_pred.append(sent_pred)
    return all_true, all_pred


In [13]:
for ds_name in ["wnut17", "scierc"]:
    test_ds_check = load_from_disk(str(TOKENIZED_DIR / ds_name / "test"))
    check_truncation(test_ds_check, f"{ds_name}/test")
print()
print("As in Step 1.5: any sequences hitting the limit here means a truncated test sentence")
print("could look like the model 'missed' an entity that was never actually shown to it.")


wnut17/test: max length = 198, 0 / 1287 sequences hit the 256-token limit (0.00%)
scierc/test: max length = 135, 0 / 551 sequences hit the 256-token limit (0.00%)

As in Step 1.5: any sequences hitting the limit here means a truncated test sentence
could look like the model 'missed' an entity that was never actually shown to it.


In [14]:
def cross_domain_boundary_eval(dataset_name, model, tokenizer, conll_id2label):
    # Load this dataset's OWN tokenized test set and its OWN label map --
    # the "labels" field in this dataset uses THIS dataset's label ids, not CoNLL's.
    test_ds = load_from_disk(str(TOKENIZED_DIR / dataset_name / "test"))
    with open(LABELS_DIR / f"{dataset_name}_label_map.json") as f:
        target_label_map = json.load(f)
    target_id2label = {int(k): v for k, v in target_label_map["id2label"].items()}

    true_ids, pred_ids = predict_labels(model, tokenizer, test_ds, conll_id2label)

    # Decode gold labels using the TARGET dataset's id2label, and predictions using CoNLL's --
    # they're different numeric spaces, so each must be decoded with its own mapping.
    true_tags = [[target_id2label[i] for i in row] for row in true_ids]
    pred_tags = [[conll_id2label[i] for i in row] for row in pred_ids]

    # Collapse both to boundary-only (B-ENT / I-ENT / O) before scoring
    true_boundary = [[collapse_to_boundary(t) for t in row] for row in true_tags]
    pred_boundary = [[collapse_to_boundary(t) for t in row] for row in pred_tags]

    return {
        "dataset": dataset_name,
        "precision": precision_score(true_boundary, pred_boundary),
        "recall": recall_score(true_boundary, pred_boundary),
        "f1": f1_score(true_boundary, pred_boundary),
    }

wnut_zero_shot = cross_domain_boundary_eval("wnut17", model, tokenizer, conll_id2label)
scierc_zero_shot = cross_domain_boundary_eval("scierc", model, tokenizer, conll_id2label)

print("Zero-shot cross-domain, entity-boundary-only F1 (CoNLL-2003-trained baseline):\n")
for r in (wnut_zero_shot, scierc_zero_shot):
    print(f"  {r['dataset']:10s}  P={r['precision']:.3f}  R={r['recall']:.3f}  F1={r['f1']:.3f}")


Zero-shot cross-domain, entity-boundary-only F1 (CoNLL-2003-trained baseline):

  wnut17      P=0.603  R=0.616  F1=0.610
  scierc      P=0.223  R=0.040  F1=0.068


## Step 9 — Save results

These numbers feed directly into your Results section and the Cost vs Performance slide.


In [15]:
RESULTS_DIR = PROCESSED / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

baseline_results = {
    "in_domain_conll2003_test": test_results,
    "zero_shot_cross_domain_boundary_f1": {
        "wnut17": wnut_zero_shot,
        "scierc": scierc_zero_shot,
    },
}

with open(RESULTS_DIR / "baseline_results.json", "w") as f:
    json.dump(baseline_results, f, indent=2, default=str)

print("Saved to:", RESULTS_DIR / "baseline_results.json")


Saved to: /content/drive/MyDrive/AAI590/data/processed/results/baseline_results.json
